In [ ]:
# In the name of GOD, the Most Gracious, the Most Merciful

In [ ]:
import numpy as np
import math
import sys
import json
import pandas as pd
import time
import pickle
import scipy.stats
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d 
import matplotlib.tri as mtri
from math import factorial
from json import JSONEncoder
import time
import random
from deap import creator, base, tools, algorithms
from deap.benchmarks.tools import diversity, convergence
import itertools
import fptools as fp
from itertools import groupby
import operator
import copy
from operator import attrgetter, itemgetter

In [ ]:
def dataSettoList(dateSet):
    List=[]
    for idx in range (0,dateSet.size()):
        List.append(dateSet.getY(idx))
    return List

In [ ]:
from javaconnector import KronosModelConnector
    
kronos = KronosModelConnector()

In [ ]:
# converted to multi-objective
def generateIndividual(num_old_vessels, OfficerminTime, OfficerminFlow, SailorminTime, SailorminFlow, MaxPeriod, 
                       MaxFlow, OfficerlowRec, OfficerhighRec, SailorlowRec, SailorhighRec, MinNumNew, MaxNumNew, 
                       TimeMinRetOld, TimeMinAcqNew, transition_length_ratio, average_operations_time, kronos):
    
    
    
    
    directory_file='conf/Model-Config_OMEGA_n2_new.xlsx'        
    
    kronos.load_config(directory_file)                         
    
    
    #Generate Officer flow time 
    Oft=np.random.randint(OfficerminTime,MaxPeriod, size=(4, 4))
    
    #Generate officer personnel flow amount 
    Opf=np.random.randint(OfficerminFlow,MaxFlow, size=(4, 4))
    
    #Generate sailor flow time 
    Sft=np.random.randint(SailorminTime,MaxPeriod, size=(4, 4))
    #Generate sailor personnel flow 
    Spf=np.random.randint(SailorminTime,MaxFlow, size=(4, 4))
    
    
    #############Rank 4 flows are ZERO
    
    Oft[1,3], Oft[3,3], Sft[1,3], Sft[3,3]=0,0,0,0
   
    Opf[1,3], Opf[3,3], Spf[1,3], Spf[3,3]=0,0,0,0
    
    
    #Generate officer and sialor recruitment rate
    
    num_quarters=int(kronos.conf.sim_length_months/3)
    
    
    #num_old_vessels=6 #this is given always 
    
    
    ORec=np.random.randint(OfficerlowRec, OfficerhighRec, size=(num_quarters))
    SRec=np.random.randint(SailorlowRec, SailorhighRec, size=(num_quarters))
    
    #Generate retirement time of old vessels; acquisition time of new vessels; size of new vessels
    
    #we need transition length and end of simulation ratio for 30+10 years it is equal to 0.75
    
    
    transition_length=int(transition_length_ratio*num_quarters) #0.75 equal to 30 years 
    
    NumNewVessels=random.randint(MinNumNew, MaxNumNew)
    
    TimeRetOld=random.sample(range(TimeMinRetOld, transition_length, average_operations_time), num_old_vessels) 
                                                                                          
    TimeAcqNew=random.sample(range(TimeMinAcqNew, transition_length, average_operations_time), NumNewVessels)
                
    return [Oft.tolist(), Opf.tolist() , Sft.tolist(), Spf.tolist(), ORec.tolist(), SRec.tolist(), list(TimeRetOld), list(TimeAcqNew)]


In [ ]:
################################
OfficerminTime=2 
OfficerminFlow=0 
SailorminTime=4 
SailorminFlow=0 
MaxPeriod=26
MaxFlow=25
OfficerlowRec=0
OfficerhighRec=13
SailorlowRec=0
SailorhighRec=90

###Vessel renewal parameters
num_old_vessels=6
MinNumNew=12
MaxNumNew=14
TimeMinRetOld=0
TimeMinAcqNew=0
transition_length_ratio=0.75
average_operations_time=2 #dont change this without updating excel

#################################

In [ ]:
policy = generateIndividual(num_old_vessels, OfficerminTime, OfficerminFlow, SailorminTime, SailorminFlow, MaxPeriod, 
                       MaxFlow, OfficerlowRec, OfficerhighRec, SailorlowRec, SailorhighRec, MinNumNew, MaxNumNew, 
                       TimeMinRetOld, TimeMinAcqNew, transition_length_ratio, average_operations_time, kronos)

In [ ]:
print("policy=", policy)

In [ ]:
## fleet size 
def Size_fleet_time(policy, kronos):
    
    num_quarters=int(kronos.conf.sim_length_months/3)
    
    new_vessel=np.zeros(int(num_quarters))
    
    for vessel in policy[-1]:
        
        a_inactive=np.zeros(max(0,vessel))
        
        a_active=np.ones(min(num_quarters-vessel,num_quarters))
        
        new_vessel=new_vessel+np.concatenate((a_inactive, a_active),axis=0)
        
    
    old_vessel=np.zeros(int(num_quarters))
    
    for vessel in policy[-2]:
        
        a_inactive=np.zeros(min(num_quarters-vessel,num_quarters))
        
        a_active=np.ones(max(0,vessel))
        
        old_vessel=old_vessel+np.concatenate((a_active, a_inactive),axis=0)
        
        
    fleet_size_time=new_vessel+old_vessel
    
    return new_vessel, old_vessel, fleet_size_time

In [ ]:
B_old=42*(10**6)
B_new=47*(10**6)

discountrate=0.02/4 
epsilon_workforce=0.005/4 
epsilon_asset=0.01/4 

Mu_old=0.04/4
Sigma_old=0.005
Mu_new=0.03/4
Sigma_new=0.0015
       
Po=58*(10**8)

In [ ]:
## Calculating AMC 
def AssetCost(B_old, B_new, Mu_old, Sigma_old, Mu_new, Sigma_new, discountrate, Po, epsilon_asset, kronos, policy):
    
    OM_Cost_New=0
    OM_Cost_Old=0
    Purcahse_Cost=0
    TotalCost=0
    
    
    List_of_OM_Cost_New=[]
    List_of_Purchase_Cost_New=[]
    List_of_OM_Cost_Old=[]

#####Truncated Normal distribution of O&M cost growth rate 
    
    Gama_old=np.absolute(random.normalvariate(Mu_old, Sigma_old))
    Gama_new=np.absolute(random.normalvariate(Mu_new, Sigma_new))

    
    num_quarters=int(kronos.conf.sim_length_months/3)
    
    
    #Calculating New Vessels O&M and Purchasing Costs
    
    ### O&M Costs of New Vessels
    
    for j in policy[-1]:
        k=[]
        for t in range(j,num_quarters+1):
            p=((1/((1+discountrate)**(t)))*(B_new*math.exp(Gama_new*(t-j))))
            OM_Cost_New+=p  
            k.append(p) 
            
        ####This list is created for plotting the O&M cost of each new vessel during planning horizon              
        List_of_OM_Cost_New.append(k)
                                     
    ### Purchase Costs of New Vessels
    
    ########## Price of buying new vessels###################
    Price=[]
    for t in policy[-1]:
        Price.append(Po*((1+discountrate+epsilon_asset)**t))
    ########################################################
    
    for n in range(len(policy[-1])):
        
        pc=((1/((1+discountrate)**(policy[-1][n])))*Price[n])
        
        Purcahse_Cost+=pc 
        
        ####This list is created for plotting the purchase cost of new vessels 
        List_of_Purchase_Cost_New.append(pc)
                                     
    
    ### O&M Costs of Old Vessels  
    
    #######Time acquision of old vessels#####  
    TimeAcqOld=np.array([-24,-23,-21,-20,-19,-17])*4 #convert years to quarters
    #########################################
    
    for i in range(len(policy[-2])):
        l=[]             
        for t in range(policy[-2][i]): 
            m=(1/((1+discountrate)**t))*(B_old*math.exp(Gama_old*(t-TimeAcqOld[i])))
            OM_Cost_Old+=m
            l.append(m) 
            
        ####This list is created for plotting the O&M cost of each old vessel during planning horizon       
        List_of_OM_Cost_Old.append(l)
            
    TotalCost=OM_Cost_New+Purcahse_Cost+OM_Cost_Old                         
    
    
    return TotalCost, OM_Cost_New, Purcahse_Cost, OM_Cost_Old, List_of_OM_Cost_New, List_of_Purchase_Cost_New, List_of_OM_Cost_Old  
 

In [ ]:
# converted to multi-objective
############ Mutation functions
def simple_mutation_sailor(SailorminTime, SailorminFlow, MaxPeriod, MaxFlow, policy):
    #Sailor time flow of amount
    #decide one or two points will be mutated######
    flow_or_amount=np.random.randint(2,4)
   
    which_rank=np.random.randint(4)
    
    if which_rank!=3:
        which_flow=np.random.randint(4)
    else:
        which_flow=np.random.choice([0,2])
    
    if flow_or_amount==2:
        policy[flow_or_amount][which_flow] [which_rank]=np.random.randint(SailorminTime,MaxPeriod)
    else:
        policy[flow_or_amount][which_flow] [which_rank]=np.random.randint(SailorminFlow,MaxFlow)
        
    return policy,
    
def simple_mutation_officer(OfficerminTime, OfficerminFlow, MaxPeriod, MaxFlow, policy):
    #need to check if it is taking the right policy
    #office time flow or amount random 0-1
    flow_or_amount=np.random.randint(2)
    #takes rank from 1-4 0-3
    which_rank=np.random.randint(4)
    #takes flow type 0-3
    if which_rank!=3:
        which_flow=np.random.randint(4)
    else:
        which_flow=np.random.choice([0,2])
    
    if flow_or_amount==0:
        policy[flow_or_amount][which_flow][which_rank]=np.random.randint(OfficerminTime,MaxPeriod)
    else:
        policy[flow_or_amount] [which_flow][which_rank]=np.random.randint(OfficerminFlow,MaxFlow)
    
    
    return policy,


def simple_mutation_officerRecruitment(OfficerlowRec, OfficerhighRec, policy):
    #Officer recruitment amount 6-13
    
    nORec=len(policy[-4])

    which_recruitment=np.random.randint(nORec)
    policy[-4][which_recruitment]=np.random.randint(OfficerlowRec, OfficerhighRec+1)
    
        
    return policy, 
    
    
def simple_mutation_sailorRecruitment(SailorlowRec, SailorhighRec, policy):
    #Sailor recruitment amount 14-83 
    
    nSRec=len(policy[-3])
        
    which_recruitment=np.random.randint(nSRec)
    policy[-3][which_recruitment]=np.random.randint(SailorlowRec,SailorhighRec+1)
    
        
    return policy,

def simple_mutation_TimeRetOld(TimeMinRetOld, transition_length_ratio, average_operations_time, kronos, policy):
    
    nTRetOld=len(policy[-2])
        
    which_TRetOld=np.random.randint(nTRetOld)
    
    num_quarters=int(kronos.conf.sim_length_months/3)
    
    transition_length=int(transition_length_ratio*num_quarters)  
    
    policy[-2][which_TRetOld]=random.sample(range(TimeMinRetOld, transition_length, average_operations_time), 1)[0]
    
        
    return policy,

def simple_mutation_TimeAcqNew(TimeMinAcqNew,transition_length_ratio, average_operations_time, kronos, policy):
    
    nTAcqNew=len(policy[-1])
        
    which_TAcqNew=np.random.randint(nTAcqNew)

    num_quarters=int(kronos.conf.sim_length_months/3)
    
    transition_length=int(transition_length_ratio*num_quarters) 
    
    policy[-1][which_TAcqNew]=random.sample(range(TimeMinAcqNew, transition_length, average_operations_time), 1)[0]
        
    return policy,


def choice_mutation(OfficerminTime, OfficerminFlow, SailorminTime, SailorminFlow, MaxPeriod, MaxFlow, OfficerlowRec, 
                    OfficerhighRec, SailorlowRec, SailorhighRec, TimeMinRetOld, 
                    TimeMinAcqNew, transition_length_ratio, average_operations_time, kronos, 
                    policy):

    weight= random.choice(range(6))
    
    if weight==0:

        return simple_mutation_officer(OfficerminTime, OfficerminFlow, MaxPeriod, MaxFlow, policy)
    
    elif weight==1:
        
        return simple_mutation_sailor(SailorminTime, SailorminFlow, MaxPeriod, MaxFlow, policy)
    
    elif weight==2:
        
        return simple_mutation_officerRecruitment(OfficerlowRec, OfficerhighRec, policy)

    elif weight==3:
        
        return simple_mutation_sailorRecruitment(SailorlowRec, SailorhighRec, policy)
    
    elif weight==4:
            
        return simple_mutation_TimeRetOld(TimeMinRetOld, transition_length_ratio, average_operations_time, kronos, policy)
    
    elif weight==5:
        
        return simple_mutation_TimeAcqNew(TimeMinAcqNew, transition_length_ratio, average_operations_time, kronos, policy)
    
    else:
        
        raise Exception("Mutation Error")

In [ ]:
########## Simulation
def Fitness_Cvar_MultiObj(kronos,  average_operations_time, costSurplus, 
                          B_old, B_new, Mu_old, Sigma_old, Mu_new, Sigma_new,discountrate, Po,
                          epsilon_workforce, epsilon_asset, 
                          var_level, lamda, replication, policy):
    
      
    #lamda is risk aversion set between 0-1
    #0 risk averse
    #var_level is between 0-100 usully set as 90-95
    
    #var_levels and lamda now lists with 3 elements corresponding to each objective
    
    
    directory_file='conf/Model-Config_OMEGA_n2_new.xlsx'                
    
    kronos.load_config(directory_file)                                 
   
    workforceTypes=["Officer", "Sailor"]
    flows=["Reset to NSGL", "NSGL to Available","Reset to Readying","Readying to Available"]
    ranks=["Rank 1", "Rank 2","Rank 3","Rank 4"]
    
    
    ######################Decision Variables##############################
    i, j, k=0,0,0
    #i type
    #j flow
    #k rank
    for types in workforceTypes:
        j=0
        for flow in flows:
            k=0
            for rank in ranks:
                kronos.conf.set_average_time( types, flow, rank,  policy[i][j][k])
                kronos.conf.set_max_transfer( types, flow, rank, policy[i+1][j][k] )
                
                k=k+1
            j=j+1
        i=i+2
    ##############realisation of stochastic variables comes##############################
    
    num_quarters=int(kronos.conf.sim_length_months/3)
    
    
    kronos.conf.clear_input_rate_schedules()
    

    ########set recruitment from policy###############
    
    for i in range(0,num_quarters):
        kronos.conf.add_input_rate_schedule(i, "Sailor", policy[-3][i])
        kronos.conf.add_input_rate_schedule(i, "Officer", policy[-4][i])
        
   

    kronos.conf.clear_fleet_schedules()
    
    
    #size asset function has to be called here
    
    kronos.conf.set_average_operations_time(average_operations_time)
    
    #kronos may not be needed to call
    #=============================================
    SizeAsset=Size_fleet_time(policy, kronos)
    
    for i in range(0, num_quarters, average_operations_time):
        
        kronos.conf.adjust_fleet_size(i, int(SizeAsset[2][i])) #sizeasset should come from function
             
    #=======================================
    
    
    
    TotalCost=[] #list of workforce cost each replication
    TotalAvail=[] #list of total not deployed platforms
    TotalLifeCycleCost=[] #list of total life-cycle cost each replication 
    
    critical_ratio=(1+discountrate+epsilon_workforce)/(1+discountrate)
    
    salary_discounted_increase=np.array([critical_ratio**t for t in range(num_quarters)])
    
    #define normal random parameters for each rank and status here
    
    Officer_Rank1_mu=0.09
    Officer_Rank2_mu=0.07
    Officer_Rank3_mu=0.03
    Officer_Rank4_mu=0.02
    
    Officer_Rank1_std=Officer_Rank1_mu/5.0
    Officer_Rank2_std=Officer_Rank2_mu/5.0
    Officer_Rank3_std=Officer_Rank3_mu/5.0
    Officer_Rank4_std=Officer_Rank4_mu/5.0
    
    Sailor_Rank1_mu=0.16
    Sailor_Rank2_mu=0.13
    Sailor_Rank3_mu=0.05
    Sailor_Rank4_mu=0.04
    
    Sailor_Rank1_std=Sailor_Rank1_mu/5.0
    Sailor_Rank2_std=Sailor_Rank2_mu/5.0
    Sailor_Rank3_std=Sailor_Rank3_mu/5.0
    Sailor_Rank4_std=Sailor_Rank4_mu/5.0
    
        ########set loss as stochastic###############
    for i in range(replication):  
        
        
        kronos.conf.set_loss_wastage( "Officer", "Reset", "Rank 1", np.absolute(random.normalvariate(Officer_Rank1_mu, Officer_Rank1_std)) )
        kronos.conf.set_loss_wastage( "Officer", "Reset", "Rank 2", np.absolute(random.normalvariate(Officer_Rank2_mu, Officer_Rank2_std)) )
        kronos.conf.set_loss_wastage( "Officer", "Reset", "Rank 3", np.absolute(random.normalvariate(Officer_Rank3_mu, Officer_Rank3_std)) )
        kronos.conf.set_loss_wastage( "Officer", "Reset", "Rank 4", np.absolute(random.normalvariate(Officer_Rank4_mu, Officer_Rank4_std)) )
    
        kronos.conf.set_loss_wastage( "Officer", "NSGL", "Rank 4", np.absolute(random.normalvariate(Officer_Rank4_mu, Officer_Rank4_std)) )
        
        
        kronos.conf.set_loss_wastage( "Officer", "Readying", "Rank 1", np.absolute(random.normalvariate(Officer_Rank1_mu, Officer_Rank1_std)))
        kronos.conf.set_loss_wastage( "Officer", "Readying", "Rank 2", np.absolute(random.normalvariate(Officer_Rank2_mu, Officer_Rank2_std))) 
        kronos.conf.set_loss_wastage( "Officer", "Readying", "Rank 3", np.absolute(random.normalvariate(Officer_Rank3_mu, Officer_Rank3_std)))
        
        
        #####################################
        kronos.conf.set_loss_wastage( "Sailor", "Reset", "Rank 1", np.absolute(random.normalvariate(Sailor_Rank1_mu, Sailor_Rank1_std)))
        kronos.conf.set_loss_wastage( "Sailor", "Reset", "Rank 2", np.absolute(random.normalvariate(Sailor_Rank2_mu, Sailor_Rank2_std)))
        kronos.conf.set_loss_wastage( "Sailor", "Reset", "Rank 3", np.absolute(random.normalvariate(Sailor_Rank3_mu, Sailor_Rank3_std)))
        kronos.conf.set_loss_wastage( "Sailor", "Reset", "Rank 4", np.absolute(random.normalvariate(Sailor_Rank4_mu, Sailor_Rank4_std)))
        
        kronos.conf.set_loss_wastage( "Sailor", "NSGL", "Rank 4", np.absolute(random.normalvariate(Sailor_Rank4_mu, Sailor_Rank4_std)))
        
        kronos.conf.set_loss_wastage( "Sailor", "Readying", "Rank 1", np.absolute(random.normalvariate(Sailor_Rank1_mu, Sailor_Rank1_std)) )
        kronos.conf.set_loss_wastage( "Sailor", "Readying", "Rank 2", np.absolute(random.normalvariate(Sailor_Rank2_mu, Sailor_Rank2_std)))
        kronos.conf.set_loss_wastage( "Sailor", "Readying", "Rank 3", np.absolute(random.normalvariate(Sailor_Rank3_mu, Sailor_Rank3_std)) )
        
        
        #####################################
    
        kronos.init_model()             
                
        kronos.conf.set_slot_mode( "ZERO_AVAILABILITY" )            
                    
        kronos.main.write_output = False                            
             
        kronos.run_model()                                             
    
        Sailor1_Gap=np.array(dataSettoList(kronos.main.get_workforce( "Sailor" ).rank_1_gapDS))
        Sailor2_Gap=np.array(dataSettoList(kronos.main.get_workforce( "Sailor" ).rank_2_gapDS))
        Sailor3_Gap=np.array(dataSettoList(kronos.main.get_workforce( "Sailor" ).rank_3_gapDS))
        Sailor4_Gap=np.array(dataSettoList(kronos.main.get_workforce( "Sailor" ).rank_4_gapDS))
        Officer1_Gap=np.array(dataSettoList(kronos.main.get_workforce( "Officer" ).rank_1_gapDS))
        Officer2_Gap=np.array(dataSettoList(kronos.main.get_workforce( "Officer" ).rank_2_gapDS))
        Officer3_Gap=np.array(dataSettoList(kronos.main.get_workforce( "Officer" ).rank_3_gapDS))
        Officer4_Gap=np.array(dataSettoList(kronos.main.get_workforce( "Officer" ).rank_4_gapDS))
    
        #############workforce cost calculation######################
        
        TotalSurplusCost=0
    
        gaps=[Sailor1_Gap,Sailor2_Gap,Sailor3_Gap,Sailor4_Gap, Officer1_Gap,Officer2_Gap,Officer3_Gap,Officer4_Gap]
        
    
        for gap, salary in zip(gaps, costSurplus):
    
            updated_salary=salary*salary_discounted_increase
        
            TotalSurplusCost+=np.sum((gap*updated_salary).clip(min=0))
    
        ###########################################################
    
        docked_platforms=sum(np.array(dataSettoList(kronos.main.docked_platformsDS)))
    
        TotalCost.append(TotalSurplusCost)
        
        TotalAvail.append(docked_platforms)
        
        ACost,_,_,_,_,_,_=AssetCost(B_old, B_new, Mu_old, Sigma_old, Mu_new, Sigma_new, discountrate, Po, epsilon_asset
                                    ,kronos, policy)
    
        TotalLifeCycleCost.append(ACost)
    
    
    TotalCost=np.array(TotalCost)
    varCost = np.percentile(TotalCost, var_level[0])
    cvarCost = TotalCost[TotalCost >= varCost].mean() 
    
    TotalAvail=np.array(TotalAvail)
    varAvail = np.percentile(TotalAvail, var_level[1])
    cvarAvail = TotalAvail[TotalAvail >= varAvail].mean() 
    
    TotalLifeCycleCost=np.array(TotalLifeCycleCost)
    varLifeCycleCost = np.percentile(TotalLifeCycleCost, var_level[2])
    cvarLifeCycleCost = TotalLifeCycleCost[TotalLifeCycleCost >= varLifeCycleCost].mean() 
    
    
    
    return lamda[0]*np.mean(TotalCost)+ (1-lamda[0])*cvarCost, lamda[1]*np.mean(TotalAvail)+ (1-lamda[1])*cvarAvail, lamda[2]*np.mean(TotalLifeCycleCost)+(1-lamda[2])*cvarLifeCycleCost


In [ ]:
########### RL

def RL(offspring, CXPB, MUTPB, alpha0, gamma, epsilon0, decay_a, decay_e, state, g, q_table, NES, Elite_Solutions, mean_WC_pop0, mean_CG_pop0, mean_CSC_pop0, min_WC_pop0, min_CG_pop0, min_CSC_pop0, max_WC_pop0, max_CG_pop0, max_CSC_pop0, minimums_first_front_pop, maximums_first_front_pop, min_crowding_distance_pop, max_crowding_distance_pop, min_crowding_distance_pop0, max_crowding_distance_pop0):
    
   
    ######## First action #######################################
      
    def a1(): # first OnePoint crossover second our mutation 
        
        
        toolbox = base.Toolbox()
        
        # register the goal / fitness function
        toolbox.register("evaluate", Fitness_Cvar_MultiObj, 
                     kronos,  average_operations_time, costSurplus, 
                          B_old, B_new, Mu_old, Sigma_old, Mu_new, Sigma_new,discountrate, Po,
                          epsilon_workforce, epsilon_asset, 
                          var_level, lamda, replication)
    
        # register the crossover operator (cxOnePoint) 
        toolbox.register("mate", tools.cxOnePoint)
    
        # register a mutation operator (our mutation function)  
        toolbox.register("mutate", choice_mutation, OfficerminTime, OfficerminFlow, SailorminTime, SailorminFlow, MaxPeriod, 
                     MaxFlow, OfficerlowRec, OfficerhighRec, SailorlowRec, SailorhighRec, TimeMinRetOld, 
                    TimeMinAcqNew, transition_length_ratio, average_operations_time, kronos)
        # Apply crossover and mutation on the offspring
        
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values
                
                    
        # Evaluate the individuals with an invalid fitness
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit
            
        
        ########################### Updating Elite Solutions #################################
        Pool_solutions =[]
        Pool_solutions.extend(offspring)
        Pool_solutions.extend(Elite_Solutions)
        
        # Sorting solutions of Pool_solutions into different nondomination levels
        nondominations =tools.sortLogNondominated(Pool_solutions, len(Pool_solutions), first_front_only=False)
    
        # assigning crowding distance to solutions of different fronts in nondominations     
        for ft in nondominations:
            assignCrowdingDist(ft)
            
            
        # Updating Elite Solutions 
        i=0
        Updated_Elite_Solutions =[]
        count = int(NES)
        
        while len(Updated_Elite_Solutions) < NES:
            
            if len(nondominations[i]) == count:
                Updated_Elite_Solutions.extend(nondominations[i])
            elif len(nondominations[i]) > count: 
                sorted_nondomination = sorted(nondominations[i], key=attrgetter("fitness.crowding_dist"), reverse=True)
                Updated_Elite_Solutions.extend(sorted_nondomination[:count])
            elif len(nondominations[i]) < count: 
                Updated_Elite_Solutions.extend(nondominations[i])
                count = count - len(nondominations[i])
            i = i + 1        
        ###############################################################################
        
        ##################### finding statistics of fitness values for offspring-for state####################
        # Gather all the fitnesses in one list and compute the stats
        fits_offspring = (ind.fitness.values for ind in offspring)
        fits_offspring_t = zip(*fits_offspring)             # Transpose fitnesses for analysis
 
        length = len(offspring)
        sums = map(sum, fits_offspring_t)
        means_offspring = [sum_ / length for sum_ in sums] 
        ################################################################################################################
        
        ######################## calculating reward ######################################################
        
        # Sorting solutions of offspring into different nondomination levels
        nondominations_r =tools.sortLogNondominated(offspring, len(offspring), first_front_only=False)
    
        # assigning crowding distance to solutions of the first front in nondominations     
        for ft in nondominations_r:
            assignCrowdingDist(ft)
        
            
        ############# fitness part
        fits_first_front_offspring = (ind.fitness.values for ind in nondominations_r[0])
        fits_first_front_offspring_t = zip(*fits_first_front_offspring)     # Transpose fitnesses for analysis
        minimums_first_front_offspring = list(map(min, fits_first_front_offspring_t)) 
        
        fits_first_front_offspring2 = (ind.fitness.values for ind in nondominations_r[0])
        fits_first_front_offspring_t2 = zip(*fits_first_front_offspring2)     # Transpose fitnesses for analysis
        maximums_first_front_offspring = list(map(max, fits_first_front_offspring_t2))
        
        # determining 4 points
        minimums_WC_offspring = minimums_first_front_offspring[0]
        minimums_CG_offspring = minimums_first_front_offspring[1]
        minimums_CSC_offspring = minimums_first_front_offspring[2]
        
        maximums_WC_offspring = maximums_first_front_offspring[0]
        maximums_CG_offspring = maximums_first_front_offspring[1]
        maximums_CSC_offspring = maximums_first_front_offspring[2]
        
        minimums_WC_pop = minimums_first_front_pop[0]
        minimums_CG_pop = minimums_first_front_pop[1]
        minimums_CSC_pop = minimums_first_front_pop[2]
        
        maximums_WC_pop = maximums_first_front_pop[0]
        maximums_CG_pop = maximums_first_front_pop[1]
        maximums_CSC_pop = maximums_first_front_pop[2]
        
        # normalising 4 points
        norm_min_WC_offspring = minimums_WC_offspring / min_WC_pop0 
        norm_min_CG_offspring = minimums_CG_offspring / min_CG_pop0
        norm_min_CSC_offspring = minimums_CSC_offspring / min_CSC_pop0
        
        norm_max_WC_offspring = maximums_WC_offspring / max_WC_pop0
        norm_max_CG_offspring = maximums_CG_offspring / max_CG_pop0
        norm_max_CSC_offspring = maximums_CSC_offspring / max_CSC_pop0
        
        norm_min_WC_pop = minimums_WC_pop / min_WC_pop0 
        norm_min_CG_pop = minimums_CG_pop / min_CG_pop0
        norm_min_CSC_pop = minimums_CSC_pop / min_CSC_pop0
        
        norm_max_WC_pop = maximums_WC_pop / max_WC_pop0
        norm_max_CG_pop = maximums_CG_pop / max_CG_pop0
        norm_max_CSC_pop = maximums_CSC_pop / max_CSC_pop0
        
        # calculating distances between points
        #min
        if norm_min_WC_offspring-norm_min_WC_pop <0:
            d_min_WC = norm_min_WC_offspring-norm_min_WC_pop
        else:
            d_min_WC =0
        
        if norm_min_CG_offspring-norm_min_CG_pop <0:
            d_min_CG = norm_min_CG_offspring-norm_min_CG_pop
        else:
            d_min_CG = 0
            
        if norm_min_CSC_offspring-norm_min_CSC_pop <0:
            d_min_CSC = norm_min_CSC_offspring-norm_min_CSC_pop
        else:
            d_min_CSC = 0
        #max    
        if norm_max_WC_offspring-norm_max_WC_pop <0:
            d_max_WC = norm_max_WC_offspring-norm_max_WC_pop
        else:
            d_max_WC =0
        
        if norm_max_CG_offspring-norm_max_CG_pop <0:
            d_max_CG = norm_max_CG_offspring-norm_max_CG_pop
        else:
            d_max_CG = 0
            
        if norm_max_CSC_offspring-norm_max_CSC_pop <0:
            d_max_CSC = norm_max_CSC_offspring-norm_max_CSC_pop
        else:
            d_max_CSC = 0
                
        
        Distance_min_points = math.sqrt((d_min_WC**2) + (d_min_CG**2) + (d_min_CSC**2)) 
        Distance_max_points = math.sqrt((d_max_WC**2) + (d_max_CG**2) + (d_max_CSC**2)) 
        
        # fiteness reward
        fitness_reward = Distance_min_points + Distance_max_points 
        
        
        ############# diversity part
        merged = list(itertools.chain.from_iterable(nondominations_r))
        sorted_nondomination_offspring = sorted(merged, key=attrgetter("fitness.crowding_dist"), reverse=True)
        
        min_crowding_distance_offspring = sorted_nondomination_offspring[-1].fitness.crowding_dist 
        max_crowding_distance_offspring = sorted_nondomination_offspring[0].fitness.crowding_dist 
        
        if math.isinf(min_crowding_distance_offspring):
            min_crowding_distance_offspring = 20
        if math.isinf(max_crowding_distance_offspring):
            max_crowding_distance_offspring = 20
            
        
        if min_crowding_distance_offspring - min_crowding_distance_pop > 0:
            d_min_crowding_distance = (min_crowding_distance_offspring - min_crowding_distance_pop) / min_crowding_distance_pop0
        else:
            d_min_crowding_distance = 0
            
        if max_crowding_distance_offspring - max_crowding_distance_pop > 0:
            d_max_crowding_distance = (max_crowding_distance_offspring - max_crowding_distance_pop) / max_crowding_distance_pop0
        else:
            d_max_crowding_distance = 0
            
        diversity_reward = d_min_crowding_distance + d_max_crowding_distance
        
        reward_a1= fitness_reward + diversity_reward
        
        return reward_a1, means_offspring, invalid_ind, offspring, Updated_Elite_Solutions
    
    ######## Second action #######################################
    
    def a2(): # first Two Point crossover second our mutation 
        
        
        toolbox = base.Toolbox()
        
        # register the goal / fitness function
        toolbox.register("evaluate", Fitness_Cvar_MultiObj, 
                     kronos,  average_operations_time, costSurplus, 
                          B_old, B_new, Mu_old, Sigma_old, Mu_new, Sigma_new,discountrate, Po,
                          epsilon_workforce, epsilon_asset, 
                          var_level, lamda, replication)
    
        # register the crossover operator (cxOrdered) 
        toolbox.register("mate", tools.cxTwoPoint)
    
        # register a mutation operator (our mutation function)  
        toolbox.register("mutate", choice_mutation, OfficerminTime, OfficerminFlow, SailorminTime, SailorminFlow, MaxPeriod, 
                     MaxFlow, OfficerlowRec, OfficerhighRec, SailorlowRec, SailorhighRec, TimeMinRetOld, 
                    TimeMinAcqNew, transition_length_ratio, average_operations_time, kronos)
        
        # Apply crossover and mutation on the offspring
        
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values
                
                    
        # Evaluate the individuals with an invalid fitness
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit
              
        ########################### Updating Elite Solutions #################################
        Pool_solutions =[]
        Pool_solutions.extend(offspring)
        Pool_solutions.extend(Elite_Solutions)
        
        # Sorting solutions of Pool_solutions into different nondomination levels
        nondominations =tools.sortLogNondominated(Pool_solutions, len(Pool_solutions), first_front_only=False)
    
        # assigning crowding distance to solutions of different fronts in nondominations     
        for ft in nondominations:
            assignCrowdingDist(ft)
            
        # Updating Elite Solutions 
        i=0
        Updated_Elite_Solutions =[]
        count = int(NES)
        
        while len(Updated_Elite_Solutions) < NES:
            
            if len(nondominations[i]) == count:
                Updated_Elite_Solutions.extend(nondominations[i])
            elif len(nondominations[i]) > count: 
                sorted_nondomination = sorted(nondominations[i], key=attrgetter("fitness.crowding_dist"), reverse=True)
                Updated_Elite_Solutions.extend(sorted_nondomination[:count])
            elif len(nondominations[i]) < count: 
                Updated_Elite_Solutions.extend(nondominations[i])
                count = count - len(nondominations[i])
            i = i + 1         
        ###############################################################################
        
        ##################### finding statistics of fitness values for offspring-for state ####################
        # Gather all the fitnesses in one list and compute the stats
        fits_offspring = (ind.fitness.values for ind in offspring)
        fits_offspring_t = zip(*fits_offspring)             # Transpose fitnesses for analysis

        length = len(offspring)
        sums = map(sum, fits_offspring_t)
        means_offspring = [sum_ / length for sum_ in sums] 
        ################################################################################################################

        ######################## calculating reward ######################################################
        
        # Sorting solutions of offspring into different nondomination levels
        nondominations_r =tools.sortLogNondominated(offspring, len(offspring), first_front_only=False)
    
        # assigning crowding distance to solutions of the first front in nondominations     
        for ft in nondominations_r:
            assignCrowdingDist(ft)
        
            
        ############# fitness part
        fits_first_front_offspring = (ind.fitness.values for ind in nondominations_r[0])
        fits_first_front_offspring_t = zip(*fits_first_front_offspring)     # Transpose fitnesses for analysis
        minimums_first_front_offspring = list(map(min, fits_first_front_offspring_t)) 
        
        fits_first_front_offspring2 = (ind.fitness.values for ind in nondominations_r[0])
        fits_first_front_offspring_t2 = zip(*fits_first_front_offspring2)     # Transpose fitnesses for analysis
        maximums_first_front_offspring = list(map(max, fits_first_front_offspring_t2))
        
        # determining 4 points
        minimums_WC_offspring = minimums_first_front_offspring[0]
        minimums_CG_offspring = minimums_first_front_offspring[1]
        minimums_CSC_offspring = minimums_first_front_offspring[2]
        
        maximums_WC_offspring = maximums_first_front_offspring[0]
        maximums_CG_offspring = maximums_first_front_offspring[1]
        maximums_CSC_offspring = maximums_first_front_offspring[2]
        
        minimums_WC_pop = minimums_first_front_pop[0]
        minimums_CG_pop = minimums_first_front_pop[1]
        minimums_CSC_pop = minimums_first_front_pop[2]
        
        maximums_WC_pop = maximums_first_front_pop[0]
        maximums_CG_pop = maximums_first_front_pop[1]
        maximums_CSC_pop = maximums_first_front_pop[2]
        
        # normalising 4 points
        norm_min_WC_offspring = minimums_WC_offspring / min_WC_pop0 
        norm_min_CG_offspring = minimums_CG_offspring / min_CG_pop0
        norm_min_CSC_offspring = minimums_CSC_offspring / min_CSC_pop0
        
        norm_max_WC_offspring = maximums_WC_offspring / max_WC_pop0
        norm_max_CG_offspring = maximums_CG_offspring / max_CG_pop0
        norm_max_CSC_offspring = maximums_CSC_offspring / max_CSC_pop0
        
        norm_min_WC_pop = minimums_WC_pop / min_WC_pop0 
        norm_min_CG_pop = minimums_CG_pop / min_CG_pop0
        norm_min_CSC_pop = minimums_CSC_pop / min_CSC_pop0
        
        norm_max_WC_pop = maximums_WC_pop / max_WC_pop0
        norm_max_CG_pop = maximums_CG_pop / max_CG_pop0
        norm_max_CSC_pop = maximums_CSC_pop / max_CSC_pop0
        
        # calculating distances between points
        #min
        if norm_min_WC_offspring-norm_min_WC_pop <0:
            d_min_WC = norm_min_WC_offspring-norm_min_WC_pop
        else:
            d_min_WC =0
        
        if norm_min_CG_offspring-norm_min_CG_pop <0:
            d_min_CG = norm_min_CG_offspring-norm_min_CG_pop
        else:
            d_min_CG = 0
            
        if norm_min_CSC_offspring-norm_min_CSC_pop <0:
            d_min_CSC = norm_min_CSC_offspring-norm_min_CSC_pop
        else:
            d_min_CSC = 0
        #max    
        if norm_max_WC_offspring-norm_max_WC_pop <0:
            d_max_WC = norm_max_WC_offspring-norm_max_WC_pop
        else:
            d_max_WC =0
        
        if norm_max_CG_offspring-norm_max_CG_pop <0:
            d_max_CG = norm_max_CG_offspring-norm_max_CG_pop
        else:
            d_max_CG = 0
            
        if norm_max_CSC_offspring-norm_max_CSC_pop <0:
            d_max_CSC = norm_max_CSC_offspring-norm_max_CSC_pop
        else:
            d_max_CSC = 0
                
        
        Distance_min_points = math.sqrt((d_min_WC**2) + (d_min_CG**2) + (d_min_CSC**2)) 
        Distance_max_points = math.sqrt((d_max_WC**2) + (d_max_CG**2) + (d_max_CSC**2)) 
        
        # fiteness reward
        fitness_reward = Distance_min_points + Distance_max_points 
        
        
        ############# diversity part
        merged = list(itertools.chain.from_iterable(nondominations_r))
        sorted_nondomination_offspring = sorted(merged, key=attrgetter("fitness.crowding_dist"), reverse=True)
        
        min_crowding_distance_offspring = sorted_nondomination_offspring[-1].fitness.crowding_dist 
        max_crowding_distance_offspring = sorted_nondomination_offspring[0].fitness.crowding_dist 
        
        if math.isinf(min_crowding_distance_offspring):
            min_crowding_distance_offspring = 20
        if math.isinf(max_crowding_distance_offspring):
            max_crowding_distance_offspring = 20
            
        
        if min_crowding_distance_offspring - min_crowding_distance_pop > 0:
            d_min_crowding_distance = (min_crowding_distance_offspring - min_crowding_distance_pop) / min_crowding_distance_pop0
        else:
            d_min_crowding_distance = 0
            
        if max_crowding_distance_offspring - max_crowding_distance_pop > 0:
            d_max_crowding_distance = (max_crowding_distance_offspring - max_crowding_distance_pop) / max_crowding_distance_pop0
        else:
            d_max_crowding_distance = 0
            
        diversity_reward = d_min_crowding_distance + d_max_crowding_distance
        
        reward_a2= fitness_reward + diversity_reward
        
        return reward_a2, means_offspring, invalid_ind, offspring, Updated_Elite_Solutions
    
    ######## Third action #######################################
    
    def a3(): # first Uniform crossover second our mutation 
        
        
        toolbox = base.Toolbox()
        
        # register the goal / fitness function
        toolbox.register("evaluate", Fitness_Cvar_MultiObj, 
                     kronos,  average_operations_time, costSurplus, 
                          B_old, B_new, Mu_old, Sigma_old, Mu_new, Sigma_new,discountrate, Po,
                          epsilon_workforce, epsilon_asset, 
                          var_level, lamda, replication)
        
        # register the crossover operator (cxPartialyMatched) 
    
        toolbox.register("mate", tools.cxUniform, indpb=0.7)
    
        # register a mutation operator (our mutation function)  
        toolbox.register("mutate", choice_mutation, OfficerminTime, OfficerminFlow, SailorminTime, SailorminFlow, MaxPeriod, 
                     MaxFlow, OfficerlowRec, OfficerhighRec, SailorlowRec, SailorhighRec, TimeMinRetOld, 
                    TimeMinAcqNew, transition_length_ratio, average_operations_time, kronos)
        
        # Apply crossover and mutation on the offspring
        
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CXPB:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(mutant)
                del mutant.fitness.values
                
                    
        # Evaluate the individuals with an invalid fitness
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit
        
        
        ########################### Updating Elite Solutions #################################
        Pool_solutions =[]
        Pool_solutions.extend(offspring)
        Pool_solutions.extend(Elite_Solutions)
        
        # Sorting solutions of Pool_solutions into different nondomination levels
        nondominations =tools.sortLogNondominated(Pool_solutions, len(Pool_solutions), first_front_only=False)
    
        # assigning crowding distance to solutions of different fronts in nondominations     
        for ft in nondominations:
            assignCrowdingDist(ft)
            
        # Updating Elite Solutions 
        i=0
        Updated_Elite_Solutions =[]
        count = int(NES)
        
        while len(Updated_Elite_Solutions) < NES:
            
            if len(nondominations[i]) == count:
                Updated_Elite_Solutions.extend(nondominations[i])
            elif len(nondominations[i]) > count: 
                sorted_nondomination = sorted(nondominations[i], key=attrgetter("fitness.crowding_dist"), reverse=True)
                Updated_Elite_Solutions.extend(sorted_nondomination[:count])
            elif len(nondominations[i]) < count: 
                Updated_Elite_Solutions.extend(nondominations[i])
                count = count - len(nondominations[i])
            i = i + 1
                 
        ###############################################################################
        
        ##################### finding statistics of fitness values for offspring-for state ####################
        # Gather all the fitnesses in one list and compute the stats
        fits_offspring = (ind.fitness.values for ind in offspring)
        fits_offspring_t = zip(*fits_offspring)             # Transpose fitnesses for analysis
  
        length = len(offspring)
        sums = map(sum, fits_offspring_t)
        means_offspring = [sum_ / length for sum_ in sums] 
        ################################################################################################################

        ######################## calculating reward ######################################################
        
        # Sorting solutions of offspring into different nondomination levels
        nondominations_r =tools.sortLogNondominated(offspring, len(offspring), first_front_only=False)
    
        # assigning crowding distance to solutions of the first front in nondominations     
        for ft in nondominations_r:
            assignCrowdingDist(ft)
        
            
        ############# fitness part
        fits_first_front_offspring = (ind.fitness.values for ind in nondominations_r[0])
        fits_first_front_offspring_t = zip(*fits_first_front_offspring)     # Transpose fitnesses for analysis
        minimums_first_front_offspring = list(map(min, fits_first_front_offspring_t)) 
        
        fits_first_front_offspring2 = (ind.fitness.values for ind in nondominations_r[0])
        fits_first_front_offspring_t2 = zip(*fits_first_front_offspring2)     # Transpose fitnesses for analysis
        maximums_first_front_offspring = list(map(max, fits_first_front_offspring_t2))
        
        # determining 4 points
        minimums_WC_offspring = minimums_first_front_offspring[0]
        minimums_CG_offspring = minimums_first_front_offspring[1]
        minimums_CSC_offspring = minimums_first_front_offspring[2]
        
        maximums_WC_offspring = maximums_first_front_offspring[0]
        maximums_CG_offspring = maximums_first_front_offspring[1]
        maximums_CSC_offspring = maximums_first_front_offspring[2]
        
        minimums_WC_pop = minimums_first_front_pop[0]
        minimums_CG_pop = minimums_first_front_pop[1]
        minimums_CSC_pop = minimums_first_front_pop[2]
        
        maximums_WC_pop = maximums_first_front_pop[0]
        maximums_CG_pop = maximums_first_front_pop[1]
        maximums_CSC_pop = maximums_first_front_pop[2]
        
        # normalising 4 points
        norm_min_WC_offspring = minimums_WC_offspring / min_WC_pop0 
        norm_min_CG_offspring = minimums_CG_offspring / min_CG_pop0
        norm_min_CSC_offspring = minimums_CSC_offspring / min_CSC_pop0
        
        norm_max_WC_offspring = maximums_WC_offspring / max_WC_pop0
        norm_max_CG_offspring = maximums_CG_offspring / max_CG_pop0
        norm_max_CSC_offspring = maximums_CSC_offspring / max_CSC_pop0
        
        norm_min_WC_pop = minimums_WC_pop / min_WC_pop0 
        norm_min_CG_pop = minimums_CG_pop / min_CG_pop0
        norm_min_CSC_pop = minimums_CSC_pop / min_CSC_pop0
        
        norm_max_WC_pop = maximums_WC_pop / max_WC_pop0
        norm_max_CG_pop = maximums_CG_pop / max_CG_pop0
        norm_max_CSC_pop = maximums_CSC_pop / max_CSC_pop0
        
        # calculating distances between points
        #min
        if norm_min_WC_offspring-norm_min_WC_pop <0:
            d_min_WC = norm_min_WC_offspring-norm_min_WC_pop
        else:
            d_min_WC =0
        
        if norm_min_CG_offspring-norm_min_CG_pop <0:
            d_min_CG = norm_min_CG_offspring-norm_min_CG_pop
        else:
            d_min_CG = 0
            
        if norm_min_CSC_offspring-norm_min_CSC_pop <0:
            d_min_CSC = norm_min_CSC_offspring-norm_min_CSC_pop
        else:
            d_min_CSC = 0
        #max    
        if norm_max_WC_offspring-norm_max_WC_pop <0:
            d_max_WC = norm_max_WC_offspring-norm_max_WC_pop
        else:
            d_max_WC =0
        
        if norm_max_CG_offspring-norm_max_CG_pop <0:
            d_max_CG = norm_max_CG_offspring-norm_max_CG_pop
        else:
            d_max_CG = 0
            
        if norm_max_CSC_offspring-norm_max_CSC_pop <0:
            d_max_CSC = norm_max_CSC_offspring-norm_max_CSC_pop
        else:
            d_max_CSC = 0
                
        
        Distance_min_points = math.sqrt((d_min_WC**2) + (d_min_CG**2) + (d_min_CSC**2)) 
        Distance_max_points = math.sqrt((d_max_WC**2) + (d_max_CG**2) + (d_max_CSC**2)) 
        
        # fiteness reward
        fitness_reward = Distance_min_points + Distance_max_points 
        
        
        ############# diversity part
        merged = list(itertools.chain.from_iterable(nondominations_r))
        sorted_nondomination_offspring = sorted(merged, key=attrgetter("fitness.crowding_dist"), reverse=True)
        
        min_crowding_distance_offspring = sorted_nondomination_offspring[-1].fitness.crowding_dist 
        max_crowding_distance_offspring = sorted_nondomination_offspring[0].fitness.crowding_dist 
        
        if math.isinf(min_crowding_distance_offspring):
            min_crowding_distance_offspring = 20
        if math.isinf(max_crowding_distance_offspring):
            max_crowding_distance_offspring = 20
            
        
        if min_crowding_distance_offspring - min_crowding_distance_pop > 0:
            d_min_crowding_distance = (min_crowding_distance_offspring - min_crowding_distance_pop) / min_crowding_distance_pop0
        else:
            d_min_crowding_distance = 0
            
        if max_crowding_distance_offspring - max_crowding_distance_pop > 0:
            d_max_crowding_distance = (max_crowding_distance_offspring - max_crowding_distance_pop) / max_crowding_distance_pop0
        else:
            d_max_crowding_distance = 0
            
        diversity_reward = d_min_crowding_distance + d_max_crowding_distance
        
        reward_a3= fitness_reward + diversity_reward
        
        return reward_a3, means_offspring, invalid_ind, offspring, Updated_Elite_Solutions
    
   

    ###################### RL functions ####################  
    
   
    
    possible_actions=[a1(),a2(),a3()]
    epsilon = epsilon0 / (1 + g * decay_e)
    
    def Epsilon_Greedy_Policy(state, epsilon):
        if np.random.uniform(0, 1) < epsilon:
            id=np.random.choice([0,1,2])   # Explore action space
        else:
            id=np.argmax(q_table[state])     # Exploit learned values
        return id
    
    action_id=Epsilon_Greedy_Policy(state, epsilon)
    
    
        
        
       
    def step(state, possible_actions, action_id):  
        
        # calculating the value used in determining the state
        
        mean_WC_offspring = possible_actions[action_id][1][0]
        mean_CG_offspring = possible_actions[action_id][1][1]
        mean_CSC_offspring = possible_actions[action_id][1][2]
        
        ratio_WC = mean_WC_offspring / mean_WC_pop0
        ratio_CG = mean_CG_offspring / mean_CG_pop0
        ratio_CSC = mean_CSC_offspring / mean_CSC_pop0
        
        Total_ratio = ratio_WC + ratio_CG + ratio_CSC
        
        if 0<= Total_ratio < 0.25:
            next_state=0
            reward = possible_actions[action_id][0]+15
        elif 0.25 <= Total_ratio < 1:
            next_state=1
            reward = possible_actions[action_id][0]+5
        elif 1 <= Total_ratio < 2:
            next_state=2
            reward = possible_actions[action_id][0]+2.5
        elif 2 <= Total_ratio :
            next_state=3
            reward = possible_actions[action_id][0]
        
        return reward, next_state, Total_ratio
    
    reward, next_state, Total_ratio = step(state, possible_actions, action_id)
    
    old_value = q_table[state, action_id]
    next_max = np.max(q_table[next_state])
    alpha = alpha0 / (1 + g * decay_a)
    
    ################################### outputs for GA
    # Q_value calculation    
    new_value = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
    
    
    invalid_ind = possible_actions[action_id][2]
    offspring = possible_actions[action_id][3]
    Updated_Elite_Solutions = possible_actions[action_id][4]
    
    
    return Total_ratio, invalid_ind, new_value, next_state, offspring, action_id, epsilon, alpha, Updated_Elite_Solutions
        

In [ ]:
################### Pattern Extraction and Injection

def Pattern_Extraction(theta_p, theta_r, theta_f, Elite_Solutions, HP):
    
        
    # Obtaining the length of fleet transition solution
    Len_Fleet =[]
    for subList in Elite_Solutions:
        L = len(subList[-1])+len(subList[-2])
        Len_Fleet.append(L)
        
        
    # Transform Solutions (to format applicable to FPmax)
        
    # # flatten the list of solutions 
    def flatten(li):
        return sum(([x] if not isinstance(x, list) else flatten(x) for x in li), [])
    
    Main_list_flatten=[]
    for subList in Elite_Solutions:
        flat_list = flatten(subList)
        Main_list_flatten.append(flat_list)
        
    # # giving order to list items
    Main_list_flatten_ordered=[]
    for subList in Main_list_flatten:
        list_flatten_ordered = [int(str(i+1) + str(subList[i])) for i in range(len(subList))]
        Main_list_flatten_ordered.append(list_flatten_ordered)
       
    # # dividing the main solutions to decision variables 
    Main_list_flatten_ordered_p = copy.deepcopy(Main_list_flatten_ordered)
    Main_list_flatten_ordered_r = copy.deepcopy(Main_list_flatten_ordered)
    Main_list_flatten_ordered_f = copy.deepcopy(Main_list_flatten_ordered)    
    
    # # # keeping just promotion values
    Main_list_flatten_ordered_promotion=[]
    for subList in Main_list_flatten_ordered_p:
        del subList[64:]
        Main_list_flatten_ordered_promotion.append(subList)
        
    # # # keeping just recruitment values
    Main_list_flatten_ordered_recruitment=[]
    for i in range(len(Main_list_flatten_ordered_r)):
        del Main_list_flatten_ordered_r[i][:64]
        del Main_list_flatten_ordered_r[i][-Len_Fleet[i]:]  
        Main_list_flatten_ordered_recruitment.append(Main_list_flatten_ordered_r[i])
        
    # # # keeping just fleet transition values   
    Main_list_flatten_ordered_fleet_transition=[]
    for i in range(len(Main_list_flatten_ordered_f)):
        del Main_list_flatten_ordered_f[i][:-Len_Fleet[i]]
        Main_list_flatten_ordered_fleet_transition.append(Main_list_flatten_ordered_f[i])


# Assigning name to solutions 
    solution_names = []
    for i in range(len(Elite_Solutions)):
        solution_names.append(i+1)

    # Extract Patterns 
    def FPmax(list_of_solutions, theta):
        mfpt = [iset for iset in fp.maximal_frequent_itemsets(list_of_solutions, theta)]
        return mfpt
    
    # Preparing patterns_solutions to select solutions 
    def Pattern_solution_preparation(mfpt, solution_names):
        
        # Identify the patterns happen in which Elite Solutions (in promotion, recruitment, and fleet transition)
        ListIndex=[]
        for i in range(len(mfpt)):
            ListIndex_sub=[]
            for aList in Main_list_flatten_ordered:
                check =  all(item in aList for item in mfpt[i])
                if check is True:
                    ListIndex_sub.append(1) 
                else:
                    ListIndex_sub.append(0)
            ListIndex.append(ListIndex_sub)
            
        # Find the number of patterns in each Elite Solution (in promotion, recruitment, and fleet transition)
        patterns_number_in_solutions = [sum(x) for x in zip(*ListIndex)]
        
        # Dictionary of solutions'names and their pattern numbers (in promotion, recruitment, and fleet transition)
        solutions_patternsNumber = dict(zip(solution_names, patterns_number_in_solutions)) 
        
        # Rank Elite Solutions based on the higher number of patterns in them (step 1) (in promotion, recruitment, and fleet transition)
        sorted_solutions__patternsNumber = dict(sorted(solutions_patternsNumber.items(), key=lambda item: item[1], reverse=True))
       
        # Ranking Elite Solutions (step 2)
        # # Building a list of dictionary for each decision variable (promotion, recruitment, fleet transition), 
        # # each dictionary has the same value for soltions

        by_value = operator.itemgetter(1)
        listDic_sorted = [dict(g) for k, g in groupby(sorted(sorted_solutions__patternsNumber.items(), key = by_value, reverse=True), by_value)]

        
        return listDic_sorted, sorted_solutions__patternsNumber
    
    # # maximal frequent patterns in decision variables
    mfpt_promotion = FPmax(Main_list_flatten_ordered_promotion, theta_p)
    mfpt_recruitment = FPmax(Main_list_flatten_ordered_recruitment, theta_r)
    mfpt_fleet_transition = FPmax(Main_list_flatten_ordered_fleet_transition, theta_f)   
    
    ## Getting both Prepared patterns_solutions in decision variables and Number of patterns in solutions
    Promotion_both_Prepared_Number = Pattern_solution_preparation(mfpt_promotion, solution_names)
    Recruitment_both_Prepared_Number = Pattern_solution_preparation(mfpt_recruitment, solution_names)
    Fleet_transition_both_Prepared_Number = Pattern_solution_preparation(mfpt_fleet_transition, solution_names) 
    
    ## Number of patterns in solutions
    Promotion_Number_Patterns = Promotion_both_Prepared_Number[1] 
    Recruitment_Number_Patterns = Recruitment_both_Prepared_Number[1]
    Fleet_transition_Number_Patterns = Fleet_transition_both_Prepared_Number[1]
    
    # # Prepared patterns_solutions in decision variables 
    Promotion_listDic_sorted = Promotion_both_Prepared_Number[0]
    Recruitment_listDic_sorted = Recruitment_both_Prepared_Number[0]
    Fleet_transition_listDic_sorted = Fleet_transition_both_Prepared_Number[0] 
    
    # Selecting solutions to add to population (step 1)  
        
    L = min(len(Fleet_transition_listDic_sorted), len(Recruitment_listDic_sorted), len(Promotion_listDic_sorted))
    
    Common_three=[]
    Common_ProRec_RecFlt_ProFlt=[]
    Only_Promotion_Recruitment_Fleet=[]
    for i in range(L):

        Common_three_first=Promotion_listDic_sorted[i].keys()  &  Recruitment_listDic_sorted[i].keys()  &  Fleet_transition_listDic_sorted[i].keys()  
        Common_ProRec_first=Promotion_listDic_sorted[i].keys()  &  Recruitment_listDic_sorted[i].keys() - Common_three_first
        Common_RecFlt_first=Recruitment_listDic_sorted[i].keys()  &  Fleet_transition_listDic_sorted[i].keys() - Common_three_first
        Common_ProFlt_first=Promotion_listDic_sorted[i].keys()  &  Fleet_transition_listDic_sorted[i].keys() - Common_three_first
        Common_ProRec_RecFlt_ProFlt_first = {*Common_ProRec_first, *Common_RecFlt_first, *Common_ProFlt_first} 

        Common_three.append(Common_three_first)
        Common_ProRec_RecFlt_ProFlt.append(Common_ProRec_RecFlt_ProFlt_first)

        Only_Promotion_first = Promotion_listDic_sorted[i].keys() - Common_three_first - Common_ProRec_first - Common_ProFlt_first
        Only_Recruitment_first = Recruitment_listDic_sorted[i].keys() - Common_three_first - Common_ProRec_first - Common_RecFlt_first
        Only_Fleet_Transition_first = Fleet_transition_listDic_sorted[i].keys() - Common_three_first - Common_ProFlt_first - Common_RecFlt_first
        Only_Promotion_Recruitment_Fleet_first = {*Only_Promotion_first, *Only_Recruitment_first, *Only_Fleet_Transition_first}

        Only_Promotion_Recruitment_Fleet.append(Only_Promotion_Recruitment_Fleet_first)
        
    # Selecting solutions to add to population (step 2_Venn diagram)
    selected_solutions = []
    i=0
    count = HP
    while (len(selected_solutions) < HP) and (i<L):

        # First: Triple common solutions in Venn diagram
        if len(Common_three[i]) == count:
            selected_solutions.extend(Common_three[i])
        elif len(Common_three[i]) > count:
            selected_solutions = random.sample(Common_three[i], k=count)    
        elif len(Common_three[i]) < count:
            if len(Common_three[i]) != 0:
                selected_solutions.extend(Common_three[i])

            # Second: Dual common solutions in Venn diagram
            diff1 = count - len(Common_three[i])
            if len(Common_ProRec_RecFlt_ProFlt[i]) == diff1:
                selected_solutions.extend(Common_ProRec_RecFlt_ProFlt[i])    
            elif len(Common_ProRec_RecFlt_ProFlt[i]) > diff1:
                selected_solutions.extend(random.sample(Common_ProRec_RecFlt_ProFlt[i], k=diff1)) 
            elif len(Common_ProRec_RecFlt_ProFlt[i]) < diff1:
                selected_solutions.extend(Common_ProRec_RecFlt_ProFlt[i])

                # Third: Single uncommon solutions in Venn diagram
                diff2 = diff1 - len(Common_ProRec_RecFlt_ProFlt[i])
                if len(Only_Promotion_Recruitment_Fleet[i]) == diff2:
                    selected_solutions.extend(Only_Promotion_Recruitment_Fleet[i]) 
                elif len(Only_Promotion_Recruitment_Fleet[i]) > diff2:
                    selected_solutions.extend(random.sample(Only_Promotion_Recruitment_Fleet[i], k=diff2))
                elif len(Only_Promotion_Recruitment_Fleet[i]) < diff2:
                    selected_solutions.extend(Only_Promotion_Recruitment_Fleet[i])
                    count = count - len(Only_Promotion_Recruitment_Fleet[i])
        i = i + 1
                
    if len(selected_solutions) < HP: 
        raise Exception("Sorry, it is required to consider separate sets")

            
    # finding the final solutions in Elite Solutions 
    Final_selected_solutions = []
    for num in selected_solutions:
        Final_selected_solutions.append(Elite_Solutions[num - 1])
        
    
            
    return Final_selected_solutions, selected_solutions, Promotion_Number_Patterns, Recruitment_Number_Patterns, Fleet_transition_Number_Patterns      
    

In [ ]:
def assignCrowdingDist(individuals):
    """Assign a crowding distance to each individual's fitness. The
    crowding distance can be retrieve via the :attr:`crowding_dist`
    attribute of each individual's fitness.
    """
    if len(individuals) == 0:
        return

    distances = [0.0] * len(individuals)
    crowd = [(ind.fitness.values, i) for i, ind in enumerate(individuals)]

    nobj = len(individuals[0].fitness.values)

    for i in range(nobj):
        crowd.sort(key=lambda element: element[0][i])
        distances[crowd[0][1]] = float("inf")
        distances[crowd[-1][1]] = float("inf")
        if crowd[-1][0][i] == crowd[0][0][i]:
            continue
        norm = nobj * float(crowd[-1][0][i] - crowd[0][0][i])
        for prev, cur, next in zip(crowd[:-2], crowd[1:-1], crowd[2:]):
            distances[cur[1]] += (next[0][i] - prev[0][i]) / norm

    for i, dist in enumerate(distances):
        individuals[i].fitness.crowding_dist = dist

In [ ]:
################## NSGA III

def NSGA(kronos,  NGEN, CXPB, MUTPB,costSurplus, var_level, lamda, 
                       OfficerminTime, 
                       OfficerminFlow, 
                       SailorminTime, 
                       SailorminFlow, 
                       MaxPeriod, 
                       MaxFlow, 
                       OfficerlowRec, 
                       OfficerhighRec, 
                       SailorlowRec, 
                       SailorhighRec,  
                       num_old_vessels,
                       MinNumNew, 
                       MaxNumNew, 
                       TimeMinRetOld, TimeMinAcqNew, transition_length_ratio, average_operations_time,
                       B_old, B_new, Mu_old, Sigma_old, Mu_new, Sigma_new,
                       discountrate, Po, epsilon_workforce, epsilon_asset, HP, beta, alpha0, gamma, epsilon0, decay_a, decay_e, replication=30):

    
    ##### Algorithm parameters ###########################
    NOBJ = 3
    P = 12
    H = factorial(NOBJ + P - 1) / (factorial(P) * factorial(NOBJ - 1))
    MU = int(H + (4 - H % 4))
    
    #####Create Uniform Reference Point###########
    ref_points = tools.uniform_reference_points(NOBJ, P)
    
    ######################################################
    
    
    # 1 is for maximization -1 for minimization
    
    creator.create("FitnessMax", base.Fitness, weights=(-1.0,-1.0,-1.0)) #for minimization of three objectives
    creator.create("Individual", list, fitness=creator.FitnessMax)
    
    ################Indivdual Generator#################################################
    
    def generateIndividual(num_old_vessels, OfficerminTime, OfficerminFlow, SailorminTime, SailorminFlow, MaxPeriod, 
                       MaxFlow, OfficerlowRec, OfficerhighRec, SailorlowRec, SailorhighRec, MinNumNew, MaxNumNew, 
                       TimeMinRetOld, TimeMinAcqNew, transition_length_ratio, average_operations_time, kronos):
    
    
        directory_file='conf/Model-Config_OMEGA_n2_new.xlsx'  
    
        kronos.load_config(directory_file)
    
    
        #Generate Officer flow time 
        Oft=np.random.randint(OfficerminTime,MaxPeriod, size=(4, 4))
    
        #Generate officer personnel flow amount 
        Opf=np.random.randint(OfficerminFlow,MaxFlow, size=(4, 4))
    
        #Generate sailor flow time 
        Sft=np.random.randint(SailorminTime,MaxPeriod, size=(4, 4))
        #Generate sailor personnel flow 
        Spf=np.random.randint(SailorminTime,MaxFlow, size=(4, 4))
    
    
        #############Rank 4 flows are ZERO
    
        Oft[1,3], Oft[3,3], Sft[1,3], Sft[3,3]=0,0,0,0
   
        Opf[1,3], Opf[3,3], Spf[1,3], Spf[3,3]=0,0,0,0
    
    
        #Generate officer and sialor recruitment rate
    
        num_quarters=int(kronos.conf.sim_length_months/3)
    
    
        #num_old_vessels=6 #this is given always 
    
    
        ORec=np.random.randint(OfficerlowRec, OfficerhighRec, size=(num_quarters))
        SRec=np.random.randint(SailorlowRec, SailorhighRec, size=(num_quarters))
    
        #Generate retirement time of old vessels; acquisition time of new vessels; size of new vessels
    
        #we need transition length and end of simulation ratio for 30+10 years it is equal to 0.75
    
    
        transition_length=int(transition_length_ratio*num_quarters) #0.75 equal to 30 years 
        
        NumNewVessels=random.randint(MinNumNew, MaxNumNew)
    
        TimeRetOld=random.sample(range(TimeMinRetOld, transition_length, average_operations_time), num_old_vessels) 
                                                                                          
        TimeAcqNew=random.sample(range(TimeMinAcqNew, transition_length, average_operations_time), NumNewVessels)
                
        return creator.Individual([Oft.tolist(), Opf.tolist() , Sft.tolist(), Spf.tolist(), ORec.tolist(), SRec.tolist(), list(TimeRetOld), list(TimeAcqNew)])

    
    toolbox = base.Toolbox()
    
    toolbox.register("individual", generateIndividual, 
                     num_old_vessels, OfficerminTime, OfficerminFlow, SailorminTime, SailorminFlow, MaxPeriod, 
                       MaxFlow, OfficerlowRec, OfficerhighRec, SailorlowRec, SailorhighRec, MinNumNew, MaxNumNew, 
                       TimeMinRetOld, TimeMinAcqNew, transition_length_ratio, average_operations_time, kronos)
    
     # define the population to be a list of individuals
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    # register the goal / fitness function
    toolbox.register("evaluate", Fitness_Cvar_MultiObj, 
                     kronos,  average_operations_time, costSurplus, 
                          B_old, B_new, Mu_old, Sigma_old, Mu_new, Sigma_new,discountrate, Po,
                          epsilon_workforce, epsilon_asset, 
                          var_level, lamda, replication)
    
    # register the select function
    toolbox.register("select", tools.selNSGA3, ref_points=ref_points)
    
    
    start_time = time.time() #start time
    
    # setting the first population 
    pop = toolbox.population(n=MU)
    hof = tools.ParetoFront(similar=np.array_equal)
    
    # Initialize statistics object
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean, axis=0)
    stats.register("std", np.std, axis=0)
    stats.register("min", np.min, axis=0)
    stats.register("max", np.max, axis=0)
    
    logbook = tools.Logbook()
    logbook.header = "gen", "evals", "std", "min", "avg", "max"

    # Evaluate every individuals 
    fitnesses = list(map(toolbox.evaluate, pop))
    for ind, fit in zip(pop, fitnesses):
        ind.fitness.values = fit
    
    # update hof with population
    hof.update(pop)
    
    # Compile statistics about the population
    record = stats.compile(pop)
    logbook.record(gen=0, evals=len(pop), **record)
    
    
    ########################### Creating Elite Solutions #################################
    NES = int(MU/7)  # Number of Elite Solutions
    
    # Sorting solutions of pop into different nondomination levels
    nondominations =tools.sortLogNondominated(pop, len(pop), first_front_only=False)
    
    # assigning crowding distance to solutions of different fronts in nondominations     
    for ft in nondominations:
        assignCrowdingDist(ft)
        
    # Creating Elite Solutions 
    i=0
    Elite_Solutions =[]
    count = int(NES)
    while len(Elite_Solutions) < NES:
        
        if len(nondominations[i]) == count:
            Elite_Solutions.extend(nondominations[i])
        elif len(nondominations[i]) > count: 
            sorted_nondomination = sorted(nondominations[i], key=attrgetter("fitness.crowding_dist"), reverse=True)
            Elite_Solutions.extend(sorted_nondomination[:count])
        elif len(nondominations[i]) < count: 
            Elite_Solutions.extend(nondominations[i])
            count = count - len(nondominations[i])
        i = i + 1
            
    ###############################################################################
    
    ############## statistics of fitness values for the first population-for RL state ####################
    # Gather all the fitnesses in one list and compute the stats
    fits = (ind.fitness.values for ind in pop)
    fits_t = zip(*fits)             # Transpose fitnesses for analysis
    
    length = len(pop)
    sums = map(sum, fits_t)
    means_pop0 = [sum_ / length for sum_ in sums] 
    
    mean_WC_pop0 = means_pop0[0]
    mean_CG_pop0 = means_pop0[1]
    mean_CSC_pop0 = means_pop0[2]
    ################################################################################################################
    
    
    ######## max and min of three objectives and crowding distances in the first front of population-for RL reward ####################
    fits_first_front_pop = (ind.fitness.values for ind in nondominations[0])
    fits_first_front_pop_t = zip(*fits_first_front_pop)             # Transpose fitnesses for analysis
    minimums_first_front_pop = list(map(min, fits_first_front_pop_t)) 
    
    fits_first_front_pop2 = (ind.fitness.values for ind in nondominations[0])
    fits_first_front_pop_t2 = zip(*fits_first_front_pop2)             # Transpose fitnesses for analysis
    maximums_first_front_pop = list(map(max, fits_first_front_pop_t2))  
    
    min_WC_pop0 = minimums_first_front_pop[0]
    min_CG_pop0 = minimums_first_front_pop[1]
    min_CSC_pop0 = minimums_first_front_pop[2]
    
    max_WC_pop0 = maximums_first_front_pop[0]
    max_CG_pop0 = maximums_first_front_pop[1]
    max_CSC_pop0 = maximums_first_front_pop[2]
    
    # crwoding distance
    merged = list(itertools.chain.from_iterable(nondominations))
    sorted_nondomination_pop = sorted(merged, key=attrgetter("fitness.crowding_dist"), reverse=True)
        
    min_crowding_distance_pop = sorted_nondomination_pop[-1].fitness.crowding_dist
    max_crowding_distance_pop = sorted_nondomination_pop[0].fitness.crowding_dist
    
    if math.isinf(min_crowding_distance_pop):
        min_crowding_distance_pop = 20
    if math.isinf(max_crowding_distance_pop):
        max_crowding_distance_pop = 20
            
    min_crowding_distance_pop0 = sorted_nondomination_pop[-1].fitness.crowding_dist
    max_crowding_distance_pop0 = sorted_nondomination_pop[0].fitness.crowding_dist  
    
    if math.isinf(min_crowding_distance_pop0):
        min_crowding_distance_pop0 = 20
    if math.isinf(max_crowding_distance_pop0):
        max_crowding_distance_pop0 = 20
    
    
    ################################################################################################################
    
    g=0     # initial gen
    state=0 # initial state
    
    state_size=4
    action_size=3
    
    # Initializing Q_values to zero 
    q_table = np.zeros([state_size, action_size])
    
    Q_values=[0]
    Epsilon_Values=[0.35]
    Alpha_Values=[0.2]
    state_ratio =[]
    
    # list for recording number of patterns in solutions 
    selected_solutions_pattern=[]
    Number_Patterns_Prom = []
    Number_Patterns_Rec =[]
    Number_Patterns_Flee = []
      
    # Begin the evolution    
    while (g < NGEN): 
        
        # A new generation
        g = g + 1
             
        print ("generation: ", g)
        
        # Clone the selected individuals
        offspring = list(map(toolbox.clone, pop))
        
    
        #################### RL intervention #####################
                                                                                                                                                  
        Total_ratio, invalid_ind_offspring, q_value, next_state, next_offspring, action_id, epsilon, alpha, Updated_Elite_Solutions = RL(offspring, CXPB, MUTPB, alpha0, gamma, epsilon0, decay_a, decay_e, state, g, q_table, NES, Elite_Solutions, mean_WC_pop0, mean_CG_pop0, mean_CSC_pop0, min_WC_pop0, min_CG_pop0, min_CSC_pop0, max_WC_pop0, max_CG_pop0, max_CSC_pop0, minimums_first_front_pop, maximums_first_front_pop, min_crowding_distance_pop, max_crowding_distance_pop, min_crowding_distance_pop0, max_crowding_distance_pop0)
        
        # updating q_table
        q_table[state, action_id] = q_value
        
        # updating state
        state = next_state 
        
        # updating Elite Set
        Elite_Solutions = Updated_Elite_Solutions
        
        # updating invalid_ind for compile statistics 
        invalid_ind = invalid_ind_offspring
        
        
        ################## Pattern Extraction ###################
        theta_p = int(NES / 2)     # minimum support promotion variable
        theta_r = int(NES / 2)     # minimum support recruitment variable
        theta_f = 2                # minimum support fleet transition variable
        
        if (g>10) and (g % beta) == 0:    # the condistion for extracting elite solutions

            # # calling Pattern Extraction function for solutions and their number of patterns
            Frequent_patterns_number_in_solutions = Pattern_Extraction(theta_p, theta_r, theta_f, Elite_Solutions, HP) 
            
            # solution names containing patterns
            solution_names_containing_patterns = Frequent_patterns_number_in_solutions[1]
            
            # Number of patterns in promotion
            Num_Pat_Prom = Frequent_patterns_number_in_solutions[2]
            
            # Number of patterns in recruitment
            Num_Pat_Rec = Frequent_patterns_number_in_solutions[3]
            
            # Number of patterns in fleet transition
            Num_Pat_Flee = Frequent_patterns_number_in_solutions[4]
            
            # recording values related to number of patterns in solutions
            selected_solutions_pattern.append(solution_names_containing_patterns)
            Number_Patterns_Prom.append(Num_Pat_Prom)
            Number_Patterns_Rec.append(Num_Pat_Rec)
            Number_Patterns_Flee.append(Num_Pat_Flee)
                       
            # obtaining solutions with maximal patterns         
            Solutions_Extracted_list = Frequent_patterns_number_in_solutions[0] 
            
            # changing the type of Solution Extracted from list to individual 
            Solutions_Extracted =[]
            for subList in Solutions_Extracted_list:
                sol = creator.Individual(subList)
                Solutions_Extracted.append(sol) 
                
            # Evaluate every Solutions Extracted
            fitnesses_Solutions_Extracted = list(map(toolbox.evaluate, Solutions_Extracted))
            for ind, fit in zip(Solutions_Extracted, fitnesses_Solutions_Extracted):
                ind.fitness.values = fit
            
            
            # combining extracted soltions with next_offspring to select the best MU numbers  
            next_offspring.extend(Solutions_Extracted)
            
            
            # Sorting solutions of next_offspring into different nondomination levels
            nondominations =tools.sortLogNondominated(next_offspring, len(next_offspring), first_front_only=False)
    
            # assigning crowding distance to solutions of different fronts in nondominations     
            for ft in nondominations:
                assignCrowdingDist(ft)
        
            # Creating offspring 
            i=0
            offspring =[]
            count = MU
            while len(offspring) < MU:
                
                if len(nondominations[i]) == count:
                    offspring.extend(nondominations[i])
                elif len(nondominations[i]) > count: 
                    sorted_nondomination = sorted(nondominations[i], key=attrgetter("fitness.crowding_dist"), reverse=True)
                    offspring.extend(sorted_nondomination[:count])
                elif len(nondominations[i]) < count: 
                    offspring.extend(nondominations[i])
                    count = count - len(nondominations[i])
                i = i + 1


            # Select the next generation population from parents and offspring
            pop = toolbox.select(pop + offspring, MU)
              
        else:
            # updating offspring
            offspring = next_offspring
            
            # Select the next generation population from parents and offspring
            pop = toolbox.select(pop + offspring, MU)
            
        
        # updating hof with pop    
        hof.update(pop) 
        
        # Compile statistics about the new population
        record = stats.compile(pop)
        logbook.record(gen=g, evals=len(invalid_ind), **record)
    
        
        # updating several values to send to RL for calculating reward and state 
        nondominations_p =tools.sortLogNondominated(pop, len(pop), first_front_only=False)
         
        for ft in nondominations_p:
            assignCrowdingDist(ft)
    
        
        fits_first_front_pop = (ind.fitness.values for ind in nondominations_p[0])
        fits_first_front_pop_t = zip(*fits_first_front_pop)             # Transpose fitnesses for analysis
        minimums_first_front_pop = list(map(min, fits_first_front_pop_t)) 
        
        fits_first_front_pop2 = (ind.fitness.values for ind in nondominations_p[0])
        fits_first_front_pop_t2 = zip(*fits_first_front_pop2)             # Transpose fitnesses for analysis
        maximums_first_front_pop = list(map(max, fits_first_front_pop_t2))
        
        merged = list(itertools.chain.from_iterable(nondominations_p))
        sorted_nondomination_pop = sorted(merged, key=attrgetter("fitness.crowding_dist"), reverse=True)
        
        min_crowding_distance_pop = sorted_nondomination_pop[-1].fitness.crowding_dist 
        max_crowding_distance_pop = sorted_nondomination_pop[0].fitness.crowding_dist 
        
        if math.isinf(min_crowding_distance_pop):
            min_crowding_distance_pop = 20
        if math.isinf(max_crowding_distance_pop):
            max_crowding_distance_pop = 20
        
        
        ###################################################################                                                                                                                                               max_WC_pop0, max_CG_pop0, max_CSC_pop0, 
         
        Q_values.append(q_value)
        Epsilon_Values.append(epsilon)
        Alpha_Values.append(alpha)
        state_ratio.append(Total_ratio)
        
        
        
        
        
    stop_time = time.time() - start_time
    
    return pop, logbook, hof, stats, stop_time, Q_values, Epsilon_Values, Alpha_Values, state_ratio, selected_solutions_pattern, Number_Patterns_Prom, Number_Patterns_Rec, Number_Patterns_Flee  


In [ ]:
costSurplus=[22187,24435,26722,28963,26135,28397,33329,36164]
var_level, lamda=[90, 90, 90], [1, 1, 0] 

OfficerminTime=2 
OfficerminFlow=0 
SailorminTime=4 
SailorminFlow=0 
MaxPeriod=26
MaxFlow=25
OfficerlowRec=0
OfficerhighRec=13
SailorlowRec=0
SailorhighRec=90

###Vessel renewal parameters
num_old_vessels=6
MinNumNew=12
MaxNumNew=14
TimeMinRetOld=0
TimeMinAcqNew=0
transition_length_ratio=0.75
average_operations_time=2 #dont change this without updating excel

#################################
B_old=42*(10**6)
B_new=47*(10**6)

discountrate=0.02/4 
epsilon_workforce=0.005/4 
epsilon_asset=0.01/4 

Mu_old=0.04/4
Sigma_old=0.005
Mu_new=0.03/4
Sigma_new=0.0015
       
Po=58*(10**8)

######## Pattern Extraction parameters
HP = 6              # number of solutions selected from Elite Solutions to inject into population 
beta = 8           # control the frequency of pattern extraction execution 

#############Reinforcement Learning Hyperparameters    
alpha0 = 0.5      # initial learning rate
gamma = 0.75      # discount factor  
epsilon0 = 0.5    # initial greedy probability  
decay_a = 0.003    # learning rate decay
decay_e = 0.003    # epsilon decay

##############Algorithm Parameters#############
replication=30
NGEN = 300
CXPB = 0.7
MUTPB = 0.3


pop, logbook, hof, stats, stop_time, Q_values, Epsilon_Values, Alpha_Values, state_ratio, selected_solutions_pattern, Number_Patterns_Prom, Number_Patterns_Rec, Number_Patterns_Flee = NSGA(kronos,  NGEN, CXPB, MUTPB,costSurplus, var_level, lamda, 
                       OfficerminTime, 
                       OfficerminFlow, 
                       SailorminTime, 
                       SailorminFlow, 
                       MaxPeriod, 
                       MaxFlow, 
                       OfficerlowRec, 
                       OfficerhighRec, 
                       SailorlowRec, 
                       SailorhighRec,  
                       num_old_vessels,
                       MinNumNew, 
                       MaxNumNew, 
                       TimeMinRetOld, TimeMinAcqNew, transition_length_ratio, average_operations_time,
                       B_old, B_new, Mu_old, Sigma_old, Mu_new, Sigma_new,
                       discountrate, Po, epsilon_workforce, epsilon_asset, HP, beta, alpha0, gamma, epsilon0, decay_a, decay_e, replication=30)
                       


#Best values for objective functions
front=np.array([ind.fitness.values for ind in hof])
    
#Best values for decision variables
solution=np.array([ind for ind in hof])


Results=[]
Statistics={}

####INPUTS############
Statistics["salary"]=costSurplus

####OUTPUTS##################
## RL outputs
Statistics["Q_values"]=Q_values

Statistics["Epsilon_Values"]=Epsilon_Values

Statistics["Alpha_Values"]=Alpha_Values

Statistics["state_ratio"]=state_ratio

## Pattern Extraction outputs
Statistics["solutions_containing_patterns"]=selected_solutions_pattern
Statistics["Number_Of_Patterns_Promotion"]=Number_Patterns_Prom
Statistics["Number_of_Patterns_Recruitment"]=Number_Patterns_Rec
Statistics["Number_of_Patterns_Fleet_Transition"]=Number_Patterns_Flee

## NSGA outputs

Statistics["lamda"]=lamda

Statistics["var_level"]=var_level

Statistics["run_time"]=stop_time

Statistics["pareto_front_values"]=front

Statistics["pareto_solutions"]=solution

Statistics["logbook"]=logbook


        
Results.append(Statistics)
        
Statistics={}
        

class NumpyArrayEncoder(JSONEncoder):
      def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        
        if isinstance(obj, np.int32):
            return int(obj)
        
        return JSONEncoder.default(self, obj)  
    
class npEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.int32):
            return int(obj)
        return json.JSONEncoder.default(self, obj)
    
    
with open('QLFPEM_MOSO_AMC_Risk_Averse.json', 'w') as outfile:

    json.dump(Results, outfile, cls=NumpyArrayEncoder)

In [ ]:
with open('QLFPEM_MOSO_AMC_Risk_Averse.json', 'r') as read_file: 
        
    Results=json.load(read_file)

In [ ]:
df=pd.DataFrame(Results)

In [ ]:
df.head()